# Advanced: ORACC JSON Processing

This notebook walks through how raw ORACC JSON files are downloaded, parsed into word-level CSVs, cached, and reconstructed into Python objects. It also shows how to extend the dataset with new projects. It covers the following topics:

1. **ORACC JSON Stucture** — explore the original format of the data
2. **From JSON to Word CSVs** — understand how the data was transformed to a more human-friendly and assyriology-friendly format as an intermediary step, and learn what information is kept from the JSON files
3. **How to Download New Projects** — learn how to use the package code to download and process projects that were not included in the Zenodo repo (primarily Sumerian projects or projects that were added after the creation of the Zenodo repo)

In [1]:
# Install oracc-parser from PyPI
!pip install oracc-parser -q

# --- Optional: persist downloaded data across Colab sessions using Google Drive ---
# Without this, data downloads to /content/oracc_data and is lost when the session ends.
# Uncomment the lines below to mount your Drive and store data there persistently.
#
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["ORACC_DATA_DIR"] = "/content/drive/MyDrive/oracc_data"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 3.9 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
from oracc_parser.download.fetch_data import fetch_data
from oracc_parser.settings import JSONZIP_DIR

# This notebook requires the raw ORACC JSON ZIPs.
# They are not downloaded by default — run this cell once to fetch them.
if not any(JSONZIP_DIR.glob('*.zip')):
    print('Downloading ORACC JSON ZIPs from Zenodo (large download, may take a while)...')
    fetch_data(include_json_zips=True)
else:
    n = sum(1 for _ in JSONZIP_DIR.glob('*.zip'))
    print(f'JSON ZIPs already present: {n} project(s) in {JSONZIP_DIR}')

Fetching file list from https://zenodo.org/records/20625379...
  oracc_jsonzip_all.zip (662.2 MB)
  catalogues.zip (2.7 MB)


oracc_jsonzip_all.zip: 100%|██████████| 694M/694M [11:17<00:00, 1.03MB/s]


  ✓ catalogues.zip already extracted, skipping

📂 Extracting project ZIPs from oracc_jsonzip_all.zip ...


Extracting: 100%|██████████| 141/141 [00:15<00:00,  9.00file/s]


✅ Extracted 141 project ZIPs to /content/drive/MyDrive/oracc_data/jsonzip

✓ Done. Data is ready in /content/drive/MyDrive/oracc_data


## 1. ORACC JSON Structure

> ℹ️ This package's parser is based on [Niek Veldhuis's Computational Assyriology](https://github.com/niekveldhuis/compass/tree/master/2_1_Data_Acquisition_ORACC) notebooks for parsing data from ORACC. Our main addition is sign-level information.

Each ORACC project ZIP contains two key JSON files:

| File | Contents |
|---|---|
| `catalogue.json` | Metadata for every text in the project (provenance, period, genre, etc.) |
| `corpusjson/P*.json` | Full text of each tablet as a CDL tree |

### CDL Node Types

The CDL (Cuneiform Digital Library) tree is a nested JSON structure: document → chunk → sentence → phrase → word → sign. Most parsing work is a depth-first traversal that collects word objects.

The CDL tree uses a `"node"` field to tag each level:

| Node type | Field | Description |
|---|---|---|
| `"d"` | document / chunk | Top-level container (surface, obverse, reverse …) |
| `"l"` | lemma | A **word** — the leaf nodes that carry linguistic data |
| `"c"` | cluster | A phrase or sentence grouping |
| `"ll"` | line label | Column/line reference node |

The parser does a depth-first walk and collects every `"l"` node. Each lemma node looks like this:

```json
{
  "node": "l",
  "ref":  "P224485.1.1",
  "inst": "šarru[king]N",
  "f": {
    "form":  "LUGAL",
    "norm":  "šarru",
    "sense": "king",
    "pos":   "N",
    "lang":  "akk",
    "gdl":   [ ... ]
  }
}
```

The `"gdl"` (grapheme description list) inside `"f"` is the source of Unicode cuneiform and break information.

> Some ORACC projects have Unicode cuneiform in their data dump, others do not. When the Unicode cuneiform is missing, the package converts transliterated signs to Unicode cuneiform based on the sign list.

The codeblock below shows the process of how to get to the first word-level object of each JSON.

In [3]:
import json, zipfile
from oracc_parser.settings import JSONZIP_DIR

PROJECT = "saao/saa01"
project_slug = PROJECT.replace("/", "-")
zip_path = JSONZIP_DIR / f"{project_slug}.zip"

def _find_path_to_first_lemma(node):
    """Return the list of nodes on the path from node to the first 'l' node, or None."""
    if not isinstance(node, dict):
        return None
    if node.get("node") == "l":
        return [node]
    for child in node.get("cdl", []):
        result = _find_path_to_first_lemma(child)
        if result is not None:
            return [node] + result
    return None

def _print_path(path_nodes, root_keys, root_cdl_len):
    key_str = ", ".join(root_keys)
    print(f"(root)  keys: [{key_str}]  [cdl: {root_cdl_len} children]")
    for i, node in enumerate(path_nodes):
        prefix = "  " * (i + 1) + "└─ "
        node_type = node.get("node")
        keys = [k for k in node.keys() if k != "cdl"]
        cdl_len = len(node.get("cdl", []))
        key_str = ", ".join(keys[:8]) + (" ..." if len(keys) > 8 else "")
        cdl_info = f"  [cdl: {cdl_len} children]" if cdl_len else ""
        print(f"{prefix}node=\"{node_type}\"  keys: [{key_str}]{cdl_info}")
        if node_type == "l":
            f_keys = ", ".join(node.get("f", {}).keys())
            print(f"{'  ' * (i + 2)}└─ f: [{f_keys}]")

if not zip_path.exists():
    print(f"ZIP not found at {zip_path}")
    print("Run the setup cell above first.")
else:
    with zipfile.ZipFile(zip_path) as z:
        names = sorted(n for n in z.namelist() if n.endswith(".json") and "corpusjson" in n)
        print(f"Found {len(names)} corpus JSON files in {project_slug}.zip")
        for name in names[:3]:
            data = json.loads(z.read(name))
            top_keys = [k for k in data.keys() if k != "cdl"]
            cdl_len = len(data.get("cdl", []))
            print(f"── {name.split('/')[-1]} ──")
            path_nodes = None
            for child in data.get("cdl", []):
                path_nodes = _find_path_to_first_lemma(child)
                if path_nodes:
                    break
            if path_nodes:
                _print_path(path_nodes, top_keys, cdl_len)
            else:
                print("  (no lemma node found)")


Found 265 corpus JSON files in saao-saa01.zip
── P224395.json ──
(root)  keys: [type, project, source, license, license-url, more-info, UTC-timestamp, textid]  [cdl: 1 children]
  └─ node="c"  keys: [node, type, id]  [cdl: 3 children]
    └─ node="c"  keys: [node, type, subtype, id]  [cdl: 1 children]
      └─ node="c"  keys: [node, type, implicit, id, label]  [cdl: 176 children]
        └─ node="l"  keys: [node, frag, id, ref, inst, sig, f, props]
          └─ f: [lang, form, delim, gdl, cf, gw, sense, norm, pos, epos]
── P224403.json ──
(root)  keys: [type, project, source, license, license-url, more-info, UTC-timestamp, textid]  [cdl: 1 children]
  └─ node="c"  keys: [node, type, id]  [cdl: 3 children]
    └─ node="c"  keys: [node, type, subtype, id]  [cdl: 1 children]
      └─ node="c"  keys: [node, type, implicit, id, label]  [cdl: 48 children]
        └─ node="l"  keys: [node, frag, id, ref, inst, sig, f, props]
          └─ f: [lang, form, delim, gdl, cf, gw, sense, norm, pos, e

## 2. From JSON to Word CSVs

The full pipeline is:

```
extract_from_zip(project)
    └─> ProjectData  (project_catalogue dict  +  json_files list[dict])
            └─> parse_json_text(json_dict, config)
                    └─> TabletRecord
                            ├─ TabletMetadata   ← from catalogue.json
                            └─ TabletContent    ← from corpusjson/P*.json
                                    └─> tablet_record_to_dataframe(record)
                                            └─> save_word_csv(df, path)
```

**TabletRecord as intermediary:**

`TabletRecord` is the shared in-memory representation regardless of whether data came from a JSON ZIP or a word CSV. `parse_json_text()` builds it from the CDL tree; `word_dataframe_to_record()` rebuilds it from CSV rows. All downstream functions (`get_transliterations()`, `get_metadata_table()`, etc.) work on `TabletRecord` objects and therefore work identically on both routes.

> The word-level csvs are meant as a human-friendly intermediary for checking data quality, and if necessary adding annotations or corrections to the data.

### Full field mapping — JSON → TabletRecord → CSV

| JSON source | TabletRecord field | CSV column | Description |
|---|---|---|---|
| `catalogue.json → members → {key}` | `metadata.id_text` | `text_id` | Unique tablet identifier (P-number) |
| `catalogue.json → project` | `metadata.project` | `project` | ORACC project path (e.g. `saao/saa01`) |
| `catalogue.json → members → genre` | `metadata.genre` | catalogue CSV only | Text genre (e.g. Letter, Royal Inscription) |
| `catalogue.json → members → provenience` | `metadata.geographical_information` | catalogue CSV only | Find-spot, resolved to city name and Pleiades ID |
| `catalogue.json → members → period` | `metadata.chronological_information` | catalogue CSV only | Historical period, resolved to year range |
| computed (sequential per text) | `content.words[i].word_index` | `word_index` | Position of this word within the tablet |
| `corpusjson → node="l" → frag` | `content.words[].frag` | `frag` | Raw transliteration form (e.g. `a-na\t`, `EN-⸢ia⸣`) |
| `corpusjson → node="l" → ref` | `content.words[].ref` | `ref` | Full CDL reference string identifying word position: `text_id.line_number.word_number` (e.g. `P224395.2.1`) |
| `corpusjson → node="l" → inst` | `content.words[].inst` | `inst` | Lemmatisation instance combining lemma, English gloss, and POS (`šarru[king]N`) |
| `corpusjson → node="l" → f.form` | `content.words[].form` | `form` | Transliteration of the word without extra markings (e.g. `a-na`, `EN-ia`) |
| `corpusjson → node="l" → f.cf` | `content.words[].lemma_form` | `lemma_form` | Dictionary citation form (lemma) |
| `corpusjson → node="l" → f.sense` | `content.words[].sense` | `sense` | English gloss |
| `corpusjson → node="l" → f.norm` | `content.words[].norm` | `norm` | Reconstructed pronunciation of word (normalization) |
| `corpusjson → node="l" → f.pos` | `content.words[].raw_pos` | `raw_pos` | Part-of-speech tag (e.g. `N`, `V`, `PN`) |
| `corpusjson → node="l" → f.lang` | `content.words[].lang` | `lang` | Language code (e.g. `akk-x-neoass`) |
| `corpusjson → node="l" → ref` (2nd segment) | `content.words[].line` | `line` | Line number on the tablet, parsed from ref (`P224395.2.1` → `2`) |
| `corpusjson → node="l" → f.gdl[].v` | `content.words[].signs_reading` | `signs_reading` | Word transliteration regrouped with damage information on sign level |
| `corpusjson → node="l" → f.gdl[].status` | `content.words[].signs_break_pct` | `signs_break_pct` | Fraction of broken/missing signs in the word (0–1) |
| `corpusjson → node="l" → f.gdl[].n` | `content.words[].unicode` | `unicode` | Unicode cuneiform rendering of the word, individual signs separated by `;` (e.g. `𒀀; 𒈾`, `𒂗; 𒅀`) |
| `corpusjson → node="l" → f.gdl[].status` (raw) | `content.words[].break_info` | `break_info` | Raw break annotation string per sign, separated by `:` (e.g. `complete; complete`, `complete; damaged`) |


> ⚠️ Important note: not all ORACC projects have full lemmatization: i.e. POS-tags, lemma form, English sense, and normalization for every word in their corpus. When unknown, these fields remain empty.

See below the functions used to turn a JSON to a TabletRecord to a DataFrame

In [5]:
from oracc_parser.download.extract_jsons import extract_from_zip
from oracc_parser.parsing.parse_content import parse_json_text
from oracc_parser.metadata.populate import populate_metadata
from oracc_parser.models.tablet import TabletRecord
from oracc_parser.io.word_csv import record_to_word_dataframe
from oracc_parser import RunConfig
from oracc_parser.settings import JSONZIP_DIR

# Change PROJECT to explore a different corpus
PROJECT = "babcity"

project_slug = PROJECT.replace("/", "-")
zip_path = JSONZIP_DIR / f"{project_slug}.zip"

if not zip_path.exists():
    print(f"ZIP not found at {zip_path}")
    print("Run the setup cell above first.")
else:
    # Step 1: extract the ZIP
    project_data = extract_from_zip(PROJECT)
    print(f"Extracted: {len(project_data.json_files)} texts")
    print(f"Catalogue entries: {len(project_data.project_catalogue.get('members', {}))}")

    # Step 2: parse a single text into a TabletRecord
    # parse_json_text() returns TabletContent (words + string representations)
    # populate_metadata() returns TabletMetadata (provenance, period, genre, ...)
    # Together they form a TabletRecord
    sample_json = project_data.json_files[0]
    text_id = sample_json.get("textid") or sample_json.get("id", "?")
    catalogue_members = project_data.project_catalogue.get("members", {})
    print(f"Parsing: {text_id}")

    config = RunConfig(fetch_translations=False)
    record = TabletRecord()
    record.content = parse_json_text(sample_json, config)
    record.metadata = populate_metadata(catalogue_members.get(text_id, {}), text_id, PROJECT)

    period = record.metadata.chronological_information
    print(f"  Identifier: {record.metadata.identifier}")
    print(f"  Period:     {period.tablet_period.period_name if period and period.tablet_period else 'unknown'}")
    print(f"  Words:      {len(record.content.words)}")

    # Step 3: convert to word-level DataFrame
    df = record_to_word_dataframe(record)
    print(f"Word DataFrame: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")
    display(df.head(5))


12:28:51 | ERROR   | oracc_parser | Error reading babcity/corpusjson/P531078.json: Expecting value: line 1 column 1 (char 0)
ERROR:oracc_parser:Error reading babcity/corpusjson/P531078.json: Expecting value: line 1 column 1 (char 0)


Extracted: 224 texts
Catalogue entries: 276
Parsing: P531072
  Identifier: babcity_P531072
  Period:     Neo-Babylonian
  Words:      41
Word DataFrame: 41 rows x 17 columns
Columns: ['text_id', 'project', 'word_index', 'frag', 'ref', 'inst', 'form', 'lemma_form', 'sense', 'norm', 'raw_pos', 'lang', 'line', 'signs_reading', 'signs_break_pct', 'unicode', 'break_info']


,text_id,project,word_index,frag,ref,inst,form,lemma_form,sense,norm,raw_pos,lang,line,signs_reading,signs_break_pct,unicode,break_info
0,P531072,babcity,0,[...,P531072.3.1,u,x,,,,u,akk-x-neobab,3,[x],1.00,x,missing
1,P531072,babcity,1,LUGAL],P531072.3.2,šar[king]N,LUGAL,šarru,king,šar,N,akk-x-neobab,3,[LUGAL],1.00,𒈗,missing
2,P531072,babcity,2,⸢TIN⸣.TIR{ki},P531072.3.3,Babylon[1]SN,TIN.TIR{ki},Babylon,1,Babylon,SN,akk-x-neobab,3,⸢TIN⸣.TIR{ki},0.17,𒁷; 𒌁; 𒆠,damaged; complete; complete
3,P531072,babcity,3,⸢a⸣-[na,P531072.3.4,ana[to]PRP,a-na,ana,to,ana,PRP,akk-x-neobab,3,⸢a⸣-[na],0.75,𒀀; 𒈾,damaged; missing
4,P531072,babcity,4,...],P531072.3.5,u,x,,,,u,akk-x-neobab,3,[x],1.00,x,missing


## 3. How to Download New Projects

The Zenodo dataset covers the projects present at time of upload. To add a project not in the dataset, download it directly from the ORACC servers.

> ❗ **Warning:** The ORACC JSON format and the set of valid field values are defined by the ORACC project and can change over time. This parser was built and tested against a specific snapshot of the data. If ORACC changes their data structure, field names, or encoding conventions, some projects may not parse correctly without updating the code.

**Steps:**
1. Download the project ZIP from ORACC servers
2. Parse it into `TabletRecord` objects with `parse_project()`
3. Save as **word-level CSVs** (for storage and reuse with this package) or as **text-level output** (JSONL/CSV for downstream analysis)

> **Note on JSON caching:** `parse_project()` automatically caches parsed `TabletRecord` objects as JSON files in `enriched_data/cache/tablets/`. On subsequent calls with the same `RunConfig`, tablets are loaded from cache instead of re-parsed. The word CSVs in `enriched_data/oracc_csvs/` serve the same purpose in a more portable format — if you have word CSVs for a project, use `parse_project_from_word_csvs()` instead.

### Discovering Available Projects

Before downloading, you can check which ORACC projects are publicly available but not yet in your local dataset.

In [6]:
from oracc_parser.download.oracc_download import get_live_project_list
from oracc_parser.pipeline import reference_data
from oracc_parser.settings import WORD_CSV_DIR, JSONZIP_DIR

# Projects already in the local dataset
local_csvs = {d.name.replace("-", "/", 1) for d in WORD_CSV_DIR.iterdir() if d.is_dir()} if WORD_CSV_DIR.exists() else set()
local_zips = {f.stem.replace("-", "/", 1) for f in JSONZIP_DIR.glob("*.zip")} if JSONZIP_DIR.exists() else set()
local = local_csvs | local_zips

# Projects with no texts (umbrella projects or empty folders)
meta = reference_data.get_projects_metadata()
no_texts = set(meta.loc[meta["Is_Text_Folder_Empty"].str.strip() == "yes", "Project_Name"].str.strip())

# Fetch live list from ORACC servers (requires internet access)
print("Fetching live project list from ORACC...")
live = set(get_live_project_list())

missing = sorted((live - local) - no_texts)
print(f"{len(live)} projects on ORACC, {len(local)} in local dataset ({len(local_zips)} ZIPs, {len(local_csvs)} extracted)")
print(f"{len(missing)} projects with texts available to download:")
for p in missing:
    print(f"  {p}")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'oracc.museum.upenn.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
12:29:10 | INFO    | oracc_parser | Fetched 144 projects from ORACC
INFO:oracc_parser:Fetched 144 projects from ORACC


Fetching live project list from ORACC...
144 projects on ORACC, 141 in local dataset (141 ZIPs, 2 extracted)
19 projects with texts available to download:
  aemw/alalakh/idrimi
  dcclt
  dcclt/ebla
  dcclt/jena
  dcclt/nineveh
  dcclt/signlists
  dccmt
  dsst
  ecut
  edlex
  eisl
  iraq/iraq85
  kish
  kish/fieldmus
  kish/fieldmus/fmod
  kish/mathaffield
  nimrud
  obel
  obmc


In [9]:
from oracc_parser.download.oracc_download import download_zip
from oracc_parser.settings import JSONZIP_DIR

# Replace with the project you want to add
NEW_PROJECT = "obmc"

project_slug = NEW_PROJECT.replace("/", "-")
zip_path = JSONZIP_DIR / f"{project_slug}.zip"

if zip_path.exists():
    print(f"ZIP already present at {zip_path}")
else:
    print(f"Downloading {NEW_PROJECT} from ORACC...")
    zip_path = download_zip(NEW_PROJECT, output_dir=JSONZIP_DIR)
    print(f"Downloaded to {zip_path}")

12:29:52 | INFO    | oracc_parser | Downloading http://oracc.museum.upenn.edu/json/obmc.zip ...
INFO:oracc_parser:Downloading http://oracc.museum.upenn.edu/json/obmc.zip ...


12:29:52 | INFO    | oracc_parser | Saved: /content/drive/MyDrive/oracc_data/jsonzip/obmc.zip
INFO:oracc_parser:Saved: /content/drive/MyDrive/oracc_data/jsonzip/obmc.zip


Downloaded to /content/drive/MyDrive/oracc_data/jsonzip/obmc.zip


In [10]:
from oracc_parser import parse_project, RunConfig, get_full_flat_table
from oracc_parser.settings import WORD_CSV_DIR, OUTPUT_DIR
from oracc_parser.pipeline import records_to_word_dataframes
from oracc_parser.io.word_csv import save_word_csv
import pandas as pd

# Parse the downloaded project
# download=False because we already have the ZIP; set to True to download+parse in one step
config = RunConfig(fetch_translations=False, limit=5)  # remove limit to parse the full project
records = parse_project(NEW_PROJECT, config=config, download=False)
print(f"Parsed {len(records)} tablets from {NEW_PROJECT}")

# --- Option A: save as word-level CSVs (for reuse with parse_project_from_word_csvs) ---
project_csv_dir = WORD_CSV_DIR / project_slug
project_csv_dir.mkdir(parents=True, exist_ok=True)
word_dfs = records_to_word_dataframes(records)
for text_id, df in word_dfs.items():
    save_word_csv(df, project_csv_dir / f"{text_id}.csv")
print(f"Option A: saved {len(word_dfs)} word CSVs to {project_csv_dir}")

# --- Option B: export as flat text-level records (JSONL + CSV) ---
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
flat = get_full_flat_table(records)
jsonl_path = OUTPUT_DIR / f"{project_slug}.jsonl"
csv_path   = OUTPUT_DIR / f"{project_slug}.csv"
flat.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
flat.to_csv(csv_path, index=False)
print(f"Option B: saved {len(flat)} tablet records to {OUTPUT_DIR}")

Parsing obmc:   0%|          | 0/5 [00:00<?, ?it/s]12:30:04 | WARNING | oracc_parser | Unicode not found for reading: |bariga| (cleaned: bariga)
12:30:04 | WARNING | oracc_parser | Unicode not found for reading: |bariga| (cleaned: bariga)
Parsing obmc: 100%|██████████| 5/5 [00:02<00:00,  2.12it/s]
12:30:06 | INFO    | oracc_parser | Parsed 5 tablets from obmc
INFO:oracc_parser:Parsed 5 tablets from obmc
12:30:06 | INFO    | oracc_parser | Saved word CSV to /content/drive/MyDrive/oracc_data/oracc_csvs/obmc/P433189.csv (444 rows)
INFO:oracc_parser:Saved word CSV to /content/drive/MyDrive/oracc_data/oracc_csvs/obmc/P433189.csv (444 rows)
12:30:06 | INFO    | oracc_parser | Saved word CSV to /content/drive/MyDrive/oracc_data/oracc_csvs/obmc/P231451.csv (15 rows)
INFO:oracc_parser:Saved word CSV to /content/drive/MyDrive/oracc_data/oracc_csvs/obmc/P231451.csv (15 rows)
12:30:06 | INFO    | oracc_parser | Saved word CSV to /content/drive/MyDrive/oracc_data/oracc_csvs/obmc/P230710.csv (24 row

Parsed 5 tablets from obmc
Option A: saved 5 word CSVs to /content/drive/MyDrive/oracc_data/oracc_csvs/obmc
Option B: saved 5 tablet records to /content/drive/MyDrive/oracc_data/output
